In [ ]:
!pip install pandas openpyxl plotly

In [ ]:
import pandas as pd
from pathlib import Path
import plotly
import plotly.graph_objects as go
from plotly.offline import plot

In [ ]:
def create_sensor_dashboard(
    excel_path,
    output_html="dashboard.html",
    sheet_names=None
):
    excel_path = Path(excel_path)
    xls = pd.ExcelFile(excel_path)

    if sheet_names is None:
        sheet_names = xls.sheet_names[:2]

    def load_sheet(sheet_name):
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
        df.columns = [str(c).replace("Â°C", "°C").strip() for c in df.columns]

        if "Time (s)" not in df.columns:
            raise ValueError(f"'Time (s)' not found in sheet: {sheet_name}")

        if "Timestamp" not in df.columns:
            raise ValueError(f"'Timestamp' not found in sheet: {sheet_name}")

        df["Time (s)"] = pd.to_numeric(df["Time (s)"], errors="coerce")
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

        df = df.dropna(subset=["Time (s)"]).copy()
        df = df.sort_values("Time (s)")
        df = df.drop_duplicates(subset=["Time (s)"], keep="first")

        return df

    dfs = {name: load_sheet(name) for name in sheet_names}

    series_info = [
        ("FLIR Center 2x2 Temperature", "FLIR Center 2x2 Temp (°C)", "Temperature (°C)", "°C"),
        ("FLIR Full Average Temperature", "FLIR Full Avg Temp (°C)", "Temperature (°C)", "°C"),
        ("FLIR Masked Average Temperature", "FLIR Masked Avg Temp (°C)", "Temperature (°C)", "°C"),
        ("Air Temperature", "Ambient Temp (°C)", "Temperature (°C)", "°C"),
        ("Humidity", "Humidity (%)", "Humidity (%RH)", "%RH"),
    ]

    available_series = []
    for title, col, ytitle, unit in series_info:
        if all(col in dfs[name].columns for name in sheet_names):
            available_series.append((title, col, ytitle, unit))

    if not available_series:
        raise ValueError("No matching FLIR / Air / Humidity columns found.")

    summary_rows = []
    chart_divs = []

    for title, col, ytitle, unit in available_series:
        dfa = dfs[sheet_names[0]][["Time (s)", "Timestamp", col]].dropna(subset=[col]).copy()
        dfb = dfs[sheet_names[1]][["Time (s)", "Timestamp", col]].dropna(subset=[col]).copy()

        merged = pd.merge(
            dfa.rename(columns={
                "Timestamp": f"Timestamp__{sheet_names[0]}",
                col: f"{col}__{sheet_names[0]}"
            }),
            dfb.rename(columns={
                "Timestamp": f"Timestamp__{sheet_names[1]}",
                col: f"{col}__{sheet_names[1]}"
            }),
            on="Time (s)",
            how="inner"
        ).sort_values("Time (s)").reset_index(drop=True)

        if merged.empty:
            continue

        x_vals = merged[f"Timestamp__{sheet_names[0]}"]

        summary_rows.append({
            "Measurement Channel": title,
            f"{sheet_names[0]} Mean": merged[f"{col}__{sheet_names[0]}"].mean(),
            f"{sheet_names[1]} Mean": merged[f"{col}__{sheet_names[1]}"].mean(),
            f"Mean Difference ({sheet_names[1]} - {sheet_names[0]})": (
                merged[f"{col}__{sheet_names[1]}"] -
                merged[f"{col}__{sheet_names[0]}"]
            ).mean(),
            "Unit": unit,
            "Common Samples Compared": len(merged),
        })

        fig = go.Figure()

        for name in sheet_names:
            fig.add_trace(
                go.Scatter(
                    x=x_vals,
                    y=merged[f"{col}__{name}"],
                    mode="lines",
                    name=name,
                    customdata=merged[["Time (s)"]],
                    hovertemplate=(
                        "<b>%{fullData.name}</b><br>"
                        "Timestamp: %{x}<br>"
                        "Time (s): %{customdata[0]}<br>"
                        f"{title}: " + "%{y:.2f} " + unit +
                        "<extra></extra>"
                    )
                )
            )

        fig.update_layout(
            title=f"{title} — {sheet_names[0]} vs {sheet_names[1]}",
            xaxis_title="Timestamp",
            yaxis_title=ytitle,
            hovermode="x unified",
            template="plotly_white",
            height=520,
        )

        fig.update_xaxes(rangeslider_visible=True)

        chart_divs.append(
            plot(
                fig,
                include_plotlyjs=False,
                output_type="div",
                config={
                    "scrollZoom": True,
                    "displaylogo": False,
                    "responsive": True,
                },
            )
        )

    summary_df = pd.DataFrame(summary_rows)

    summary_fmt = summary_df.copy()
    for c in summary_fmt.columns:
        if c not in ["Measurement Channel", "Unit", "Common Samples Compared"]:
            summary_fmt[c] = summary_fmt[c].map(lambda x: f"{x:.2f}")

    table_html = summary_fmt.to_html(
        index=False,
        border=0,
        classes="summary-table"
    )

    included = ", ".join(f"<b>{t}</b>" for t, _, _, _ in available_series)

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>Sensor Comparison Dashboard</title>
<script>{plotly.offline.get_plotlyjs()}</script>
<style>
body {{
  font-family: Arial, sans-serif;
  margin: 24px;
  background: #ffffff;
  color: #111111;
}}
h1 {{
  margin: 0 0 8px 0;
}}
.subtitle {{
  margin: 0 0 24px 0;
  color: #444444;
  line-height: 1.5;
}}
.table-wrap, .chart {{
  margin-bottom: 28px;
  border: 1px solid #dddddd;
  border-radius: 10px;
  padding: 12px;
  background: #ffffff;
}}
table.summary-table {{
  border-collapse: collapse;
  width: 100%;
  font-size: 14px;
}}
.summary-table th, .summary-table td {{
  border: 1px solid #d9d9d9;
  padding: 8px 10px;
  text-align: center;
}}
.summary-table th {{
  background: #f5f5f5;
}}
</style>
</head>
<body>

<h1>Sensor Comparison Dashboard</h1>

<div class="subtitle">
Comparison between <b>{sheet_names[0]}</b> and <b>{sheet_names[1]}</b>.<br/>
Alignment is based on <b>common Time (s)</b> values.<br/>
Each variable is matched <b>independently</b>, so missing values in one channel do not remove valid data from other channels.<br/>
Included variables: {included}.<br/>
Zoom, pan, and range slider are enabled.
</div>

<div class="table-wrap">
<h2 style="margin-top:0;">Comparison Summary</h2>
{table_html}
</div>

{''.join(f'<div class="chart">{d}</div>' for d in chart_divs)}

</body>
</html>
"""

    output_html = Path(output_html)
    output_html.write_text(html, encoding="utf-8")

    print(f"Dashboard saved to: {output_html}")
    print(f"Compared sheets: {sheet_names}")
    display(summary_df)

    return summary_df

In [ ]:
file_path = r"C:\Flir\Analysis.xlsx"

summary = create_sensor_dashboard(
    excel_path=file_path,
    output_html="dashboard.html",
    sheet_names=["New", "Ref"]   # اسم شیت‌ها را اینجا بگذار
)

In [ ]:
file_path = "Analysis.xlsx"

summary = create_sensor_dashboard(
    excel_path=file_path,
    output_html="dashboard.html",
    sheet_names=["New", "Ref"]
)